In [1]:
%pip install openai langchain-community keyboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os
from openai import OpenAI
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. Setup Local Client
# Ensure LM Studio is RUNNING and the model is LOADED
client = OpenAI(
    base_url="http://127.0.0.1:1234/v1", 
    api_key="lm-studio"
)

# 2. Initialize Memory
history = ChatMessageHistory()

# 3. Hardcoded System Identity
# This prevents the AI from giving you the "I don't have memories" disclaimer
system_message = {
    "role": "system",
    "content": (
        "You are a helpful assistant. The user's name is Denzel, a 3rd-year Computer Engineering student. "
        "You remember his details and projects like PharmaSync and digital logic circuits (74LS192/74LS47). "
        "Be technical, concise, and professional."
    )
}

print("--- Chat Session Active (Type 'exit' to quit) ---")

while True:
    # Look for the input box at the VERY TOP of your VS Code window
    query = input("Type your message:")

    if query.lower() in ['exit', 'quit', 'bye']:
        print(">> Stopping session. Goodbye, Denzel!")
        break

    if not query.strip():
        continue

    # Print user input so it's visible in the cell log
    print(f"\nUSER: {query}")

    # Save to history
    history.add_user_message(query)

    # 4. Build the full payload (System + All Previous Messages)
    messages_to_send = [system_message]
    
    for msg in history.messages:
        # Convert LangChain format to OpenAI format
        role = "user" if msg.type == "human" else "assistant"
        messages_to_send.append({"role": role, "content": msg.content})

    try:
        # 5. Generate Response
        response = client.chat.completions.create(
            model="llama-3.2-1b-instruct", # Name doesn't matter for LM Studio
            messages=messages_to_send,
            temperature=0.7
        )

        ai_reply = response.choices[0].message.content.strip()
        
        # Save and Display
        history.add_ai_message(ai_reply)
        print(f"AI: {ai_reply}")
        print("-" * 50)
        
    except Exception as e:
        print(f"\n[CONNECTION ERROR]: Check LM Studio Server!\n{e}")
        break

--- Chat Session Active (Type 'exit' to quit) ---

USER: Hello Llama! You can call me Denzel, 22 years old and I'm a computer engineering student.
AI: Hello Denzel! It's great to meet you. As a Computer Engineering student, I'm familiar with your coursework, including the digital logic circuits topics such as 74LS192/74LS47. How can I assist you today? Do you have any specific questions or projects you'd like to discuss?
--------------------------------------------------

USER: If I am cascading two 74LS192 synchronous up/down counters to create a 0-99 timer, explain why connecting the 'Terminal Count Up' (TCU) of the first stage directly to the 'Clock Up' of the second stage might cause glitch-induced double-counting, and how would a Schmitt-trigger or proper synchronous clocking resolve this at high frequencies?
AI: I can provide you with an explanation.

When cascading two 74LS192 synchronous up/down counters to create a 0-99 timer, the 'Terminal Count Up' (TCU) of one counter direc